# Bag Detection with YOLOv8

Install the dependencies listed in the README before running the notebook.

In [ ]:
from getpass import getpass
from pathlib import Path
import os

from roboflow import Roboflow

api_key = os.environ.get("ROBOFLOW_API_KEY") or getpass("Roboflow API key: ")
project = Roboflow(api_key=api_key).workspace("birdstriketeam").project("bags-detection-ztrkp")
dataset = project.version(7).download("yolov8")
dataset_dir = Path(dataset.location)
data_yaml = str(dataset_dir / "data.yaml")

In [ ]:
for split in ("train", "valid", "test"):
    images = (dataset_dir / split / "images").iterdir()
    print(f"{split.capitalize()} images: {sum(path.is_file() for path in images)}")

In [ ]:
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

In [ ]:
model = YOLO("yolov8n.pt")

In [ ]:
model.train(
    data=data_yaml,
    epochs=100,
    imgsz=640,
    batch=8,
    lr0=0.01,
    mosaic=1.0,
)

In [ ]:
evaluation_dir = Path("runs/detect/bag-test").resolve()
metrics = model.val(
    data=data_yaml,
    split="test",
    project=str(evaluation_dir.parent),
    name=evaluation_dir.name,
    exist_ok=True,
    plots=True
)

print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)

In [ ]:
test_images = sorted(str(path) for path in (dataset_dir / "test" / "images").iterdir() if path.is_file())

results = model.predict(
    source=test_images,
    conf=0.25,
    save=True
)

plt.figure(figsize=(15, 10))

for i, result in enumerate(results[:9]):

    img = result.plot()
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.subplot(3, 3, i + 1)
    plt.imshow(img)
    plt.title(f"Image {i+1}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
for plot in sorted(evaluation_dir.glob("*.png")):
    if "curve" in plot.stem or plot.stem.startswith("confusion_matrix"):
        plt.figure(figsize=(8, 6))
        plt.imshow(plt.imread(plot))
        plt.title(plot.stem.replace("_", " "))
        plt.axis("off")
        plt.show()

In [ ]:
training_plot = Path(model.trainer.save_dir) / "results.png"
if training_plot.is_file():
    plt.figure(figsize=(16, 8))
    plt.imshow(plt.imread(training_plot))
    plt.title("Training Results")
    plt.axis("off")
    plt.show()